# 01 — Sample Wikipedia Parsing

This notebook demonstrates end-to-end parsing and cleaning of a small Wikipedia
XML extract.  It uses the helper modules in `src/` so you can iterate quickly
without running the full pipeline on a multi-GB dump.

## Prerequisites

```bash
pip install -r requirements.txt
```

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable when running from notebooks/

## 1 — Create a minimal sample XML dump

We embed a two-article MediaWiki XML snippet so the notebook works without
downloading a real dump first.

In [ ]:
SAMPLE_XML = """<mediawiki xmlns="http://www.mediawiki.org/xml/export-0.10/">
  <page>
    <title>Artificial intelligence</title>
    <ns>0</ns>
    <id>1</id>
    <revision>
      <id>100</id>
      <text xml:space="preserve">'''Artificial intelligence''' (AI) is intelligence demonstrated by machines.
It is the simulation of human intelligence processes by computer systems.
== History ==
The study of AI started in the 1950s with Alan Turing's foundational work.
== Applications ==
AI is used in search engines, recommendation systems, and robotics.
</text>
    </revision>
  </page>
  <page>
    <title>Machine learning</title>
    <ns>0</ns>
    <id>2</id>
    <revision>
      <id>200</id>
      <text xml:space="preserve">'''Machine learning''' (ML) is a subset of artificial intelligence.
It gives computers the ability to learn without being explicitly programmed.
== Types ==
* Supervised learning
* Unsupervised learning
* Reinforcement learning
</text>
    </revision>
  </page>
</mediawiki>"""

print("Sample XML prepared (%d chars)" % len(SAMPLE_XML))

## 2 — Parse with mwparserfromhell

We extract wikitext from each `<text>` element and use
[mwparserfromhell](https://github.com/earwig/mwparserfromhell) to strip
markup and obtain plain text.

In [ ]:
import xml.etree.ElementTree as ET
import mwparserfromhell

NS = 'http://www.mediawiki.org/xml/export-0.10/'

parsed_articles = []

root = ET.fromstring(SAMPLE_XML)
for page in root.findall(f'{{{NS}}}page'):
    ns_el  = page.find(f'{{{NS}}}ns')
    # skip non-article namespaces
    if ns_el is not None and ns_el.text != '0':
        continue

    title  = page.findtext(f'{{{NS}}}title', default='')
    page_id = page.findtext(f'{{{NS}}}id', default='')
    raw_text = page.findtext(f'.//{{{NS}}}text', default='')

    wikicode = mwparserfromhell.parse(raw_text)
    plain    = wikicode.strip_code(normalize=True, collapse=True)

    parsed_articles.append({'id': page_id, 'title': title, 'text': plain})
    print(f"  [{page_id}] {title} — {len(plain.split())} words")

print(f"\nTotal articles: {len(parsed_articles)}")

## 3 — Apply cleaning pipeline

In [ ]:
from src.preprocess import clean_text

for article in parsed_articles:
    article['text'] = clean_text(article['text'])

print("First article after cleaning:")
print("-" * 60)
print(parsed_articles[0]['text'][:500])

## 4 — Build WikiArticle objects and write JSONL

In [ ]:
import tempfile, pathlib
from src.schema import WikiArticle, write_jsonl, read_jsonl

articles = [
    WikiArticle(
        id=a['id'],
        url=f"https://en.wikipedia.org/wiki/{a['title'].replace(' ', '_')}",
        title=a['title'],
        text=a['text'],
    )
    for a in parsed_articles
]

_tmp_fd, _tmp_name = tempfile.mkstemp(suffix='.jsonl')
import os; os.close(_tmp_fd)
out_path = pathlib.Path(_tmp_name)
n = write_jsonl(iter(articles), out_path)
print(f"Wrote {n} articles to {out_path}")

## 5 — Reload and inspect with the data loader

In [ ]:
from src.data_loader import iter_articles

for rec in iter_articles(out_path):
    print(f"  {rec['title']}  |  words={rec['word_count']}  |  chars={rec['char_count']}")

# Clean up temp file
out_path.unlink(missing_ok=True)
print("Done.")

## 6 — (Optional) Load via HuggingFace datasets

If you have the full processed JSONL available, run the cell below.
It requires `pip install datasets`.

In [ ]:
# from src.data_loader import load_wiki_dataset
# ds = load_wiki_dataset('data/processed/wiki.jsonl')
# print(ds)
# ds[0]